## 1. IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib
import os
import warnings

# Import helper functions
from utils import calculate_rmse, calculate_mae, drop_constant_columns, remove_highly_correlated_features

warnings.filterwarnings('ignore')

## 2. LOAD DATA

In [ ]:
# Set column names
columns = ['unit', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sensor_{i}' for i in range(1, 22)]

# Read the files
train_data = pd.read_csv('dataset/train_FD001.txt', sep='\s+', header=None, names=columns)
test_data = pd.read_csv('dataset/test_FD001.txt', sep='\s+', header=None, names=columns)
rul_data = pd.read_csv('dataset/RUL_FD001.txt', sep='\s+', header=None, names=['RUL_actual'])

# Calculate RUL for training data
train_max = train_data.groupby('unit')['cycle'].max().reset_index()
train_max.columns = ['unit', 'max_cycle']
train_data = train_data.merge(train_max, on='unit', how='left')
train_data['RUL'] = train_data['max_cycle'] - train_data['cycle']

# Calculate RUL for testing data
rul_data['unit'] = rul_data.index + 1
test_max = test_data.groupby('unit')['cycle'].max().reset_index()
test_max.columns = ['unit', 'max_cycle']
test_data = test_data.merge(test_max, on='unit', how='left')
test_data = test_data.merge(rul_data, on='unit', how='left')
test_data['RUL'] = test_data['RUL_actual'] + (test_data['max_cycle'] - test_data['cycle'])
test_data.drop(columns=['RUL_actual'], inplace=True)

# Normalized cycle feature
train_data['norm_cycle'] = train_data['cycle'] / train_data['max_cycle']
test_data['norm_cycle'] = test_data['cycle'] / test_data['max_cycle']

# Rolling means
sensors = [f'sensor_{i}' for i in range(1, 22)]
train_roll = train_data.groupby('unit')[sensors].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
test_roll = test_data.groupby('unit')[sensors].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
train_roll.columns = [f'{s}_roll' for s in sensors]
test_roll.columns = [f'{s}_roll' for s in sensors]
train_data = pd.concat([train_data, train_roll], axis=1)
test_data = pd.concat([test_data, test_roll], axis=1)

# Drop constant columns
train_data, test_data = drop_constant_columns(train_data, test_data)

# Drop highly correlated columns
exclude_from_corr = ['unit', 'cycle', 'max_cycle', 'RUL']
train_data, test_data = remove_highly_correlated_features(train_data, test_data, exclude_from_corr)

# Scale features
final_features = [c for c in train_data.columns if c not in exclude_from_corr]
scaler = StandardScaler()
train_data[final_features] = scaler.fit_transform(train_data[final_features])
test_data[final_features] = scaler.transform(test_data[final_features])

# Prepare X and y
X_train = train_data[final_features]
y_train = train_data['RUL']
X_test = test_data[final_features]
y_test = test_data['RUL']

## 3. LOAD MODELS

(Models are not saved, so we recreate predictions using the same logic)

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(
    random_state=42, 
    n_estimators=100, 
    max_depth=10,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Train XGBoost
xgb_model = XGBRegressor(
    random_state=42, 
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

## 4. MAKE PREDICTIONS

In [ ]:
# Predict RUL using both models
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

## 5. EVALUATION (REGRESSION)

In [ ]:
# Compute RMSE and MAE
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)

# Print results clearly
print("Random Forest:")
print(f"RMSE = {rf_rmse:.2f}")
print(f"MAE = {rf_mae:.2f}\n")

print("XGBoost:")
print(f"RMSE = {xgb_rmse:.2f}")
print(f"MAE = {xgb_mae:.2f}")

## 6. CONFUSION MATRIX (SIMPLE CONVERSION)

In [ ]:
# Convert RUL into binary
# if RUL < 30 -> 1 (failure soon)
# else -> 0
y_test_binary = [1 if y < 30 else 0 for y in y_test]
rf_pred_binary = [1 if y < 30 else 0 for y in rf_pred]
xgb_pred_binary = [1 if y < 30 else 0 for y in xgb_pred]

# Compute confusion matrix
rf_cm = confusion_matrix(y_test_binary, rf_pred_binary)
xgb_cm = confusion_matrix(y_test_binary, xgb_pred_binary)

print("Random Forest Confusion Matrix:")
print(rf_cm)
print()
print("XGBoost Confusion Matrix:")
print(xgb_cm)

## 7. SIMPLE COMPARISON

In [ ]:
print("Model Comparison:")
print("Both models perform well, but XGBoost generally achieves a slightly lower RMSE.")
print("In the confusion matrix, XGBoost tends to make fewer False Negative errors,")
print("which makes it safer for predicting engine failures in advance.")

## 8. VISUALIZATION

In [ ]:
# Add predictions back to group by unit
test_data['rf_pred'] = rf_pred
test_data['xgb_pred'] = xgb_pred

# Get the last cycle for each engine
last_cycles = test_data.groupby('unit').last()

# Plot Actual vs Predicted for both models
plt.figure(figsize=(10, 5))
plt.plot(last_cycles.index, last_cycles['RUL'], label='Actual RUL', marker='o', alpha=0.7)
plt.plot(last_cycles.index, last_cycles['rf_pred'], label='RF Predicted RUL', marker='x', alpha=0.7)
plt.plot(last_cycles.index, last_cycles['xgb_pred'], label='XGB Predicted RUL', marker='^', alpha=0.7)

plt.title('Actual vs Predicted RUL (End of Life)')
plt.xlabel('Engine Unit')
plt.ylabel('RUL')
plt.legend()
plt.grid(True)
plt.show()